# Tanh from exponentials, without overflow

When you fuse an activation into a larger kernel you often write it from primitives rather than calling a library routine. Implement the hyperbolic tangent from `math.exp` only (do **not** call `math.tanh`):

$$\tanh(x) = \frac{e^{x} - e^{-x}}{e^{x} + e^{-x}}$$

Evaluated as written, both exponentials overflow for $|x| > 88$ and the result becomes `inf / inf = NaN`. Use the identity, valid for every real $x$,

$$\tanh(x) = \operatorname{sign}(x)\, \frac{1 - e^{-2|x|}}{1 + e^{-2|x|}}$$

which only ever exponentiates a non-positive number.

## Task
Write `tanh(x)` for a 1-D `float32` array of length `n` (possibly 0) and return a `float32` array of shape `(n,)`.

- Launch with a fixed grid of **2 blocks of 64 threads** and a grid-stride loop, so any `n` works (including 0, with no special case).
- Inputs range up to $\pm 10^4$. `tanh(0)` must be exactly `0`.
- Compared with relative tolerance $10^{-5}$, absolute $10^{-6}$.

## Runtime
GPU in Colab; checked here on Numba's CUDA simulator.

**Runtime → Change runtime type → GPU** before running.

Write your solution in the next cell, then run the test cell.

In [ ]:
!pip -q install "numba-cuda[cu12]"

In [ ]:
import math
import numpy as np
from numba import cuda

BLOCKS, TPB = 2, 64

def tanh(x):
    """x: float32 (n,). Returns tanh(x) as float32 (n,), built from math.exp."""
    d_x = cuda.to_device(x)
    d_out = cuda.device_array(x.shape[0], dtype=np.float32)
    tanh_kernel[BLOCKS, TPB](d_x, d_out)
    return d_out.copy_to_host()

@cuda.jit
def tanh_kernel(x, out):
    for i in range(cuda.grid(1), x.shape[0], cuda.gridsize(1)):
        v = x[i]
        # TODO: the textbook formula overflows for |v| > 88
        out[i] = (math.exp(v) - math.exp(-v)) / (math.exp(v) + math.exp(-v))


In [ ]:
# Tests (same harness and cases as the website)
_HARNESS = '"""Test harness shared by build.py (CPython) and the browser runner (Pyodide).\n\nOne file, so a test that passes at build time is graded the same way in the browser.\nValues cross JSON with small tags: ndarray, tuple, DataFrame/Series, non-finite floats.\n"""\nimport json, math\n\ntry:\n    import numpy as np\nexcept ImportError:  # pure-python problems never need it\n    np = None\n\n\ndef enc(x):\n    if np is not None and isinstance(x, np.ndarray):\n        return {\'__nd__\': enc(x.tolist()), \'dtype\': str(x.dtype)}\n    if np is not None and isinstance(x, np.generic):\n        return enc(x.item())\n    if isinstance(x, float) and not math.isfinite(x):\n        return {\'__f__\': repr(x)}\n    if isinstance(x, tuple):\n        return {\'__tup__\': [enc(v) for v in x]}\n    if isinstance(x, list):\n        return [enc(v) for v in x]\n    if isinstance(x, dict):\n        return {\'__dict__\': [[enc(k), enc(v)] for k, v in x.items()]}\n    t = type(x).__name__\n    if t == \'Tensor\':  # torch, in Colab and at build time\n        return enc(x.detach().cpu().numpy())\n    if t == \'DataFrame\':\n        # Cells go through enc() as Python values, so floats keep full precision (to_json caps at 15 digits).\n        return {\'__df__\': {\'columns\': [enc(c) for c in x.columns], \'data\': [enc(list(r)) for r in x.itertuples(index=False, name=None)]},\n                \'dtypes\': [str(d) for d in x.dtypes]}\n    if t == \'Series\':\n        return {\'__sr__\': enc(x.tolist()), \'name\': x.name}\n    return x\n\n\ndef dec(x):\n    if isinstance(x, list):\n        return [dec(v) for v in x]\n    if not isinstance(x, dict):\n        return x\n    if \'__nd__\' in x:\n        return np.array(dec(x[\'__nd__\']), dtype=x[\'dtype\'])\n    if \'__f__\' in x:\n        return float(x[\'__f__\'])\n    if \'__tup__\' in x:\n        return tuple(dec(v) for v in x[\'__tup__\'])\n    if \'__dict__\' in x:\n        return {_hashable(dec(k)): dec(v) for k, v in x[\'__dict__\']}\n    if \'__df__\' in x:\n        import pandas as pd\n        s = x[\'__df__\']\n        df = pd.DataFrame([dec(r) for r in s[\'data\']], columns=[dec(c) for c in s[\'columns\']])\n        for c, d in zip(df.columns, x[\'dtypes\']):\n            if d.startswith((\'int\', \'float\', \'bool\')):\n                df[c] = df[c].astype(d)\n        return df\n    if \'__sr__\' in x:\n        import pandas as pd\n        return pd.Series(dec(x[\'__sr__\']), name=x[\'name\'])\n    return x\n\n\ndef _hashable(k):\n    return tuple(k) if isinstance(k, list) else k\n\n\ndef same(got, exp, rtol=1e-6, atol=1e-8, path=\'output\'):\n    """(ok, why). Floats compare with tolerance, everything else exactly. Shapes must match."""\n    if type(exp).__name__ == \'DataFrame\':\n        if type(got).__name__ != \'DataFrame\':\n            return False, f\'{path}: expected a DataFrame, got {type(got).__name__}\'\n        if list(got.columns) != list(exp.columns):\n            return False, f\'{path}: columns {list(got.columns)} != expected {list(exp.columns)}\'\n        if got.shape != exp.shape:\n            return False, f\'{path}: shape {got.shape} != expected {exp.shape}\'\n        for c in exp.columns:\n            ok, why = same(got[c].tolist(), exp[c].tolist(), rtol, atol, f\'{path}[{c!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if type(exp).__name__ == \'Series\':\n        exp = exp.tolist()\n        got = got.tolist() if type(got).__name__ == \'Series\' else got\n    if np is not None and isinstance(exp, np.ndarray):\n        try:\n            g = np.asarray(got)\n        except Exception:\n            return False, f\'{path}: expected an array, got {type(got).__name__}\'\n        if g.shape != exp.shape:\n            return False, f\'{path}: shape {g.shape} != expected {exp.shape}\'\n        if exp.dtype.kind in \'fc\' or g.dtype.kind in \'fc\':\n            if g.dtype.kind not in \'fciub\':\n                return False, f\'{path}: expected numbers, got dtype {g.dtype}\'\n            bad = ~np.isclose(g.astype(complex if exp.dtype.kind == \'c\' else float), exp, rtol=rtol, atol=atol, equal_nan=True)\n            if bad.any():\n                i = tuple(int(v) for v in np.argwhere(bad)[0])\n                return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n            return True, \'\'\n        if not np.array_equal(g, exp):\n            i = tuple(int(v) for v in np.argwhere(g != exp)[0]) if g.shape else ()\n            return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n        return True, \'\'\n    if isinstance(exp, float) or (isinstance(exp, int) and isinstance(got, float) and not isinstance(exp, bool)):\n        if np is not None and isinstance(got, np.generic):\n            got = got.item()\n        if not isinstance(got, (int, float)) or isinstance(got, bool):\n            return False, f\'{path}: expected a number, got {type(got).__name__}\'\n        if math.isnan(exp) and math.isnan(got):\n            return True, \'\'\n        return (True, \'\') if math.isclose(got, exp, rel_tol=rtol, abs_tol=atol) else (False, f\'{path}: {got!r} != expected {exp!r}\')\n    if isinstance(exp, (list, tuple)):\n        if np is not None and isinstance(got, np.ndarray):\n            got = got.tolist()\n        if not isinstance(got, (list, tuple)):\n            return False, f\'{path}: expected a {type(exp).__name__}, got {type(got).__name__}\'\n        if len(got) != len(exp):\n            return False, f\'{path}: length {len(got)} != expected {len(exp)}\'\n        for i, (g, e) in enumerate(zip(got, exp)):\n            ok, why = same(g, e, rtol, atol, f\'{path}[{i}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if isinstance(exp, dict):\n        if not isinstance(got, dict):\n            return False, f\'{path}: expected a dict, got {type(got).__name__}\'\n        if set(got) != set(exp):\n            return False, f\'{path}: keys {sorted(map(str, got))} != expected {sorted(map(str, exp))}\'\n        for k in exp:\n            ok, why = same(got[k], exp[k], rtol, atol, f\'{path}[{k!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if np is not None and isinstance(got, np.generic):\n        got = got.item()\n    return (True, \'\') if got == exp else (False, f\'{path}: {got!r} != expected {exp!r}\')\n\n\ndef run_sql(query, setup):\n    """Run `setup` (CREATE/INSERT script) in a fresh in-memory db, then `query`; rows as lists."""\n    import sqlite3  # a separate package in Pyodide: only SQL problems load it\n    con = sqlite3.connect(\':memory:\')\n    try:\n        con.executescript(setup)\n        cur = con.execute(query)\n        return [[c[0] for c in cur.description], [list(r) for r in cur.fetchall()]]\n    finally:\n        con.close()\n\n\ndef show_args(args, runtime, n=600):\n    """Arguments as the learner would write them: one per line, SQL setup as-is."""\n    if runtime == \'sql\':\n        return args[0] if len(args[0]) <= n else args[0][:n] + \' …\'\n    return \'\\n\'.join(short(a, n) for a in args)\n\n\ndef short(x, n=300):\n    s = repr(x)\n    return s if len(s) <= n else s[:n] + \' …\'\n\n\ndef grade(fn, tests, runtime, rtol=1e-6, atol=1e-8):\n    """Run `fn` (a callable, or SQL text for runtime \'sql\') on each test; list of result dicts."""\n    out = []\n    for t in tests:\n        args, exp = prep(dec(t[\'a\']), runtime), dec(t[\'e\'])\n        try:\n            got = run_sql(fn, args[0]) if runtime == \'sql\' else plain(fn(*_copy(args)))\n            ok, why = same(got, exp, rtol, atol)\n        except Exception as e:  # learner code raised: report it like a failed test\n            got, ok, why = None, False, f\'{type(e).__name__}: {e}\'\n        out.append({\'ok\': ok, \'why\': why, \'got\': short(got) if not ok else \'\',\n                    \'inp\': show_args(args, runtime, 400) if not ok else \'\'})\n    return out\n\n\ndef prep(args, runtime):\n    """Arrays cross JSON as ndarrays; torch problems take tensors."""\n    if runtime == \'torch\':\n        import torch\n        return [torch.as_tensor(a) if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    if runtime == \'triton\':                                  # Colab only: kernels need CUDA tensors\n        import torch\n        return [torch.as_tensor(a).cuda() if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    return args\n\n\ndef plain(x):\n    """Framework arrays (torch, jax) -> numpy, recursively, so results compare and encode the same way."""\n    t = type(x).__name__\n    if t == \'Tensor\':\n        return x.detach().cpu().numpy()\n    if t in (\'ArrayImpl\', \'Array\') and hasattr(x, \'__array__\'):\n        return np.asarray(x)\n    if isinstance(x, tuple):\n        return tuple(plain(v) for v in x)\n    if isinstance(x, list):\n        return [plain(v) for v in x]\n    if isinstance(x, dict):\n        return {k: plain(v) for k, v in x.items()}\n    return x\n\n\ndef _copy(args):\n    # Tests reuse inputs across Run and Submit, so a function that mutates its input mustn\'t poison later cases.\n    import copy\n    return copy.deepcopy(args)\n'
_TESTS = '[{"a": [{"__nd__": [0.0, 0.5, -0.5, 2.0], "dtype": "float32"}], "e": {"__nd__": [0.0, 0.46211716532707214, -0.46211716532707214, 0.9640275835990906], "dtype": "float32"}}, {"a": [{"__nd__": [-100.0, -1.0, 1.0, 100.0], "dtype": "float32"}], "e": {"__nd__": [-1.0, -0.7615941762924194, 0.7615941762924194, 1.0], "dtype": "float32"}}, {"a": [{"__nd__": [], "dtype": "float32"}], "e": {"__nd__": [], "dtype": "float32"}}, {"a": [{"__nd__": [10000.0, -10000.0, 89.0, -89.0, 0.0], "dtype": "float32"}], "e": {"__nd__": [1.0, -1.0, 1.0, -1.0, 0.0], "dtype": "float32"}}, {"a": [{"__nd__": [9.999999747378752e-05, -9.999999747378752e-05, 0.009999999776482582, -0.009999999776482582], "dtype": "float32"}], "e": {"__nd__": [9.999999747378752e-05, -9.999999747378752e-05, 0.00999966636300087, -0.00999966636300087], "dtype": "float32"}}, {"a": [{"__nd__": [-2.405794382095337, -3.9730770587921143, -0.7450848817825317, 1.2613357305526733, 3.408139705657959, 0.3291192054748535, -1.6579419374465942, -2.3543410301208496, 2.246237277984619, 4.904349327087402, 0.8183063268661499, -3.699985980987549, -2.87479567527771, 4.800057411193848, 0.608647346496582, -5.196404457092285, -0.25108858942985535, -3.489677906036377, -1.8878642320632935, -1.4640175104141235, -2.139940023422241, 1.6601353883743286, -0.1892579197883606, -1.7682937383651733, 1.228913426399231, 2.489565849304199, -4.929069995880127, -0.7701903581619263, -2.94224214553833, -0.5194656848907471, -3.8682563304901123, 0.06207118183374405, -0.11365722119808197, -0.9130132794380188, -3.1437795162200928, -1.1885709762573242, -3.27398681640625, -4.06562614440918, 0.6743571758270264, -3.328049898147583, 3.5108883380889893, 2.1497628688812256, -5.993450164794922, 0.8163865804672241, -3.305149793624878, 0.09917166084051132, 0.1308959722518921, -5.96528959274292, -0.7002675533294678, -0.7673701047897339, 2.8860015869140625, -3.5443403720855713, 2.214125633239746, -3.2969183921813965, -0.9938727021217346, -2.5214195251464844, 4.346193790435791, 1.7046393156051636, 7.295197486877441, 1.9257491827011108, 2.534978151321411, 2.5220487117767334, -1.819834589958191, -0.21008534729480743, 4.051166534423828, -1.1896523237228394, 0.5663986206054688, -0.06367038190364838, 1.827649474143982, -1.0947262048721313, -0.45708566904067993, 0.7271443009376526, 0.30906942486763, -2.5949182510375977, 2.6873490810394287, -3.8954436779022217, -3.603346586227417, -3.847475290298462, 2.900916814804077, -1.0818251371383667, -2.913109064102173, -3.408064126968384, 1.2633934020996094, -3.1645219326019287, -3.816234588623047, 1.8419791460037231, -3.590123176574707, -0.9673144817352295, -0.0202846210449934, -1.3360060453414917, -0.16228188574314117, 4.016320705413818, -1.5506824254989624, -3.777921438217163, -5.510237216949463, -0.6142983436584473, -1.0567710399627686, 0.795272707939148, -1.3927338123321533, -1.4359153509140015, -2.163947343826294, -1.5592793226242065, 0.48068010807037354, -1.1410582065582275, 0.30132555961608887, 5.703561782836914, 1.4373544454574585, -4.728269577026367, 5.200588703155518, 1.0434311628341675, -2.82423996925354, 2.721146821975708, 0.05296683311462402, -1.8456555604934692, -1.900526762008667, -2.980295419692993, 0.14435695111751556, 3.2064499855041504, -0.975156307220459, 1.26247239112854, 5.626938343048096, -3.6437718868255615, 0.7725669741630554, -0.9193108081817627, -3.1777682304382324, -3.0775415897369385, -0.04585542902350426, 1.3015902042388916, -1.5947694778442383, 0.16317163407802582, 2.1666259765625, 0.889683723449707, 2.606010675430298, 1.384186863899231, 1.4640170335769653, 5.479902267456055, 1.8695156574249268, 0.3884133994579315, 2.129831314086914, -2.757370948791504, -1.013114333152771, 2.37966251373291, 1.8923600912094116, 4.645084857940674, 0.03141908720135689, -4.386983394622803, 5.841740608215332, 3.2786784172058105, -3.1762123107910156, 4.127470970153809, 0.09720839560031891, -5.441786289215088, -1.2610617876052856, -1.5267114639282227, 4.7725443840026855, -2.376206159591675, -0.760852038860321, -0.83028244972229, -1.136918306350708, -2.7422220706939697, 0.6572607159614563, 3.2304139137268066, 1.8719329833984375, -2.7824974060058594, -3.4494471549987793, 0.35692065954208374, -2.1197404861450195, -1.8904969692230225, -5.039083957672119, 5.851474761962891, 2.7498574256896973, -2.921721935272217, 2.724630832672119, 4.027461528778076, -7.1686601638793945, -1.646834373474121, -1.1639196872711182, 1.944763422012329, -0.36439627408981323, -0.6912940144538879, -0.17511416971683502, 5.559977054595947, 6.479941368103027, -1.574479341506958, -2.7787320613861084, 8.077659606933594, -2.9392340183258057, -1.7202060222625732, 0.10974452644586563, 1.4497387409210205, 3.087106704711914, 1.1729316711425781, -2.6171491146087646, 1.5191470384597778, 0.7497527599334717, 5.630756855010986, -0.044999394565820694, -4.010918617248535, -3.1350598335266113, 4.350459098815918, -1.6203936338424683, -6.3133978843688965, -1.7420991659164429, 4.5299493649508804e-05, 3.5664920806884766, -3.0434043407440186, 2.0000500679016113, 2.385897397994995, -2.0981650352478027, -0.5627691149711609, 5.308350563049316, 5.161454200744629, 2.566565990447998, 0.9958390593528748, 3.414928913116455, -0.42203184962272644, -0.28527918457984924, -2.581080436706543, 0.016690153628587723, -0.2460498809814453, 8.320895195007324, -0.5785294771194458, 3.8124539852142334, 3.961491823196411, -0.5594360828399658, 3.5081915855407715, -6.530440807342529, 0.2847815454006195, 2.571784257888794, -7.193595886230469, -3.477513551712036, 3.1682207584381104, -0.7710323333740234, -3.2945961952209473, -1.1207870244979858, -1.6277649402618408, 2.173341989517212, 1.3616576194763184, -0.8406850099563599, -1.9960280656814575, -0.16162699460983276, 3.9754676818847656, 1.0470783710479736, 1.9168367385864258, -0.4520447850227356, -3.875448703765869, -2.132530689239502, 1.9537097215652466, 0.3096483051776886, -2.711519718170166, -4.853611469268799, 0.33204570412635803, -1.0185930728912354, -1.0515906810760498, -0.9883498549461365, 0.8517153263092041, 0.8833431601524353, 2.886385679244995, 2.785264730453491, 3.996279239654541, 2.4190890789031982, -1.0068241357803345, -0.16280683875083923, 1.4826452732086182, -0.9308767914772034, 1.6483163833618164, 3.4608285427093506, -2.403738021850586, -5.31115198135376, 3.5809648036956787, 0.053279731422662735, 2.126009464263916, -3.9969160556793213, 5.198977470397949, 2.2158596515655518, -0.3782225251197815, -1.0559641122817993, 0.3416876494884491, -0.5001940131187439, -0.8360703587532043, -3.358799934387207, 3.78842830657959, -0.11873305588960648, 0.22390691936016083, 0.2787479758262634, -7.074652194976807, 1.581424355506897, -0.9874647855758667, -0.8676853775978088, 0.20936362445354462, -2.917426824569702, 2.5301425457000732, -2.0875937938690186, -2.7109742164611816, -2.586568832397461, 0.4762580990791321, 2.2373383045196533, -1.785159707069397, -4.53056001663208, 3.187472105026245], "dtype": "float32"}], "e": {"__nd__": [-0.9838614463806152, -0.9992921948432922, -0.6322075128555298, 0.8514319658279419, 0.9978108406066895, 0.317729115486145, -0.9299394488334656, -0.982127845287323, 0.9778619408607483, 0.9998900890350342, 0.6741470098495483, -0.998778223991394, -0.9936520457267761, 0.9998645782470703, 0.5431742072105408, -0.9999387264251709, -0.24594168365001678, -0.9981399178504944, -0.9551863670349121, -0.8984300494194031, -0.9726894497871399, 0.9302354454994202, -0.18703018128871918, -0.9434221386909485, 0.8422638773918152, 0.986333966255188, -0.9998953938484192, -0.6470401287078857, -0.9944508671760559, -0.47728753089904785, -0.999127209186554, 0.06199158728122711, -0.1131703332066536, -0.7225753664970398, -0.9962882995605469, -0.83013516664505, -0.9971380829811096, -0.9994117617607117, 0.5878387093544006, -0.9974309802055359, 0.9982171058654785, 0.9732136130332947, -0.9999875426292419, 0.6730983853340149, -0.9973107576370239, 0.09884781390428543, 0.13015347719192505, -0.9999868273735046, -0.6045375466346741, -0.6453976035118103, 0.9937922954559326, -0.9983323812484741, 0.9764108657836914, -0.9972661733627319, -0.7590088248252869, -0.98717200756073, 0.9996643662452698, 0.9359865188598633, 0.9999991059303284, 0.9583883881568909, 0.9875130653381348, 0.9871880412101746, -0.9488219022750854, -0.2070481926202774, 0.9993945360183716, -0.8304710388183594, 0.5127094984054565, -0.06358448415994644, 0.9495956301689148, -0.7985966801643372, -0.4277059733867645, 0.6213151812553406, 0.2995902895927429, -0.9889159202575684, 0.9907780885696411, -0.9991733431816101, -0.9985178709030151, -0.9990901947021484, 0.9939742088317871, -0.7938749194145203, -0.9941189289093018, -0.9978104829788208, 0.8519969582557678, -0.9964388608932495, -0.9990315437316895, 0.95098477602005, -0.9984781742095947, -0.7475217580795288, -0.0202818401157856, -0.8707095980644226, -0.16087214648723602, 0.9993508458137512, -0.9138979911804199, -0.9989544749259949, -0.9999672770500183, -0.5471457242965698, -0.784424901008606, 0.6613856554031372, -0.8837708830833435, -0.8928724527359009, -0.973953127861023, -0.915303647518158, 0.44678813219070435, -0.8147701025009155, 0.29252520203590393, 0.9999777674674988, 0.8931639194488525, -0.9998436570167542, 0.9999392032623291, 0.7792394161224365, -0.9929789900779724, 0.9913782477378845, 0.05291735753417015, -0.9513351321220398, -0.9562825560569763, -0.9948564767837524, 0.14336249232292175, 0.9967248439788818, -0.750961422920227, 0.8517442941665649, 0.9999740719795227, -0.9986329078674316, 0.6484196186065674, -0.7255712151527405, -0.9965318441390991, -0.9957636594772339, -0.045823317021131516, 0.8621319532394409, -0.9208773970603943, 0.16173873841762543, 0.9740904569625854, 0.7112374901771545, 0.9891577959060669, 0.8818853497505188, 0.8984299898147583, 0.999965250492096, 0.9535502195358276, 0.36999163031578064, 0.9721394777297974, -0.9919783473014832, -0.7670470476150513, 0.9830029010772705, 0.955578625202179, 0.9998153448104858, 0.031408753246068954, -0.9996906518936157, 0.9999831318855286, 0.997164785861969, -0.9965210556983948, 0.9994801878929138, 0.09690336138010025, -0.9999624490737915, -0.851356565952301, -0.9098601341247559, 0.9998568892478943, -0.982886016368866, -0.6415785551071167, -0.6806276440620422, -0.8133738040924072, -0.991732656955719, 0.576537549495697, 0.996877908706665, 0.9537690281867981, -0.9923700094223022, -0.9979842305183411, 0.342498779296875, -0.9715795516967773, -0.9554164409637451, -0.9999160170555115, 0.9999834895133972, 0.9918574094772339, -0.9942190647125244, 0.9914378523826599, 0.9993651509284973, -0.9999988079071045, -0.9284219145774841, -0.8223131895065308, 0.959909975528717, -0.34908050298690796, -0.5988126397132874, -0.1733458936214447, 0.9999703764915466, 0.9999952912330627, -0.9177353382110596, -0.9923125505447388, 0.9999998211860657, -0.9944174885749817, -0.9378878474235535, 0.10930605977773666, 0.8956412076950073, 0.995843768119812, 0.8252097368240356, -0.9893953800201416, 0.908548891544342, 0.6350014209747314, 0.9999743103981018, -0.044969044625759125, -0.999343752861023, -0.9962231516838074, 0.9996671676635742, -0.9246813058853149, -0.9999934434890747, -0.9404696226119995, 4.5299493649508804e-05, 0.9984046220779419, -0.9954649806022644, 0.9640311002731323, 0.9832117557525635, -0.9703449010848999, -0.510029137134552, 0.9999510049819946, 0.9999342560768127, 0.988273024559021, 0.7598411440849304, 0.9978402853012085, -0.39864078164100647, -0.2777840197086334, -0.9886066913604736, 0.016688603907823563, -0.24120192229747772, 0.9999998807907104, -0.5215957760810852, 0.9990242123603821, 0.9992756247520447, -0.5075588822364807, 0.9982074499130249, -0.999995768070221, 0.27732470631599426, 0.9883940815925598, -0.9999988675117493, -0.9980941414833069, 0.9964650869369507, -0.6475293040275574, -0.9972534775733948, -0.807842493057251, -0.9257426261901855, 0.9744317531585693, 0.8767769932746887, -0.6861717104911804, -0.9637458920478821, -0.16023413836956024, 0.9992955923080444, 0.7806679010391235, 0.9576559066772461, -0.4235783815383911, -0.999139666557312, -0.9722873568534851, 0.9606069326400757, 0.3001171052455902, -0.9912113547325134, -0.9998783469200134, 0.3203577399253845, -0.7692928910255432, -0.7824240326881409, -0.7566578388214111, 0.6919645071029663, 0.7080901861190796, 0.9937970638275146, 0.9924119710922241, 0.9993242621421814, 0.9842815399169922, -0.7644452452659607, -0.1613834649324417, 0.901962399482727, -0.7310024499893188, 0.9286262392997742, 0.9980295300483704, -0.9837954640388489, -0.9999512434005737, 0.9984501004219055, 0.05322937294840813, 0.9719286561012268, -0.999325156211853, 0.9999390244483948, 0.9764915704727173, -0.36116284132003784, -0.7841143012046814, 0.32898321747779846, -0.46226972341537476, -0.6837220788002014, -0.9975840449333191, 0.9989761710166931, -0.11817823350429535, 0.22023865580558777, 0.2717459201812744, -0.9999985694885254, 0.9188240170478821, -0.7562792301177979, -0.7001962065696716, 0.20635730028152466, -0.9941693544387817, 0.9873924851417542, -0.9697208404541016, -0.9912018179893494, -0.9887303709983826, 0.44324183464050293, 0.9774689078330994, -0.9452474117279053, -0.9997678399085999, 0.9965983629226685], "dtype": "float32"}}, {"a": [{"__nd__": [-213.0167236328125, 238.96607971191406, -10.859175682067871, -139.66998291015625, -128.00799560546875, 92.37323760986328, 177.87435913085938, -39.6340446472168, 189.38687133789062, 417.3956298828125, -41.66167068481445, -229.8893280029297, -210.88006591796875, -278.06707763671875, -127.93309020996094, -144.75584411621094, -265.0206604003906, 106.03684997558594, -522.267822265625, -264.1597900390625, 403.05035400390625, 207.56048583984375, 104.55833435058594, 91.45181274414062, 354.18743896484375, 107.12593078613281, 15.515417098999023, -246.0320281982422, -14.733363151550293, -146.77679443359375, -141.2350616455078, -142.84779357910156, -8.929372787475586, -65.9421615600586, 174.22210693359375, -265.2362976074219, -33.894039154052734, 371.8995056152344, 7.779467582702637, 239.72303771972656, 227.37269592285156, 263.3295593261719, 147.62384033203125, -276.0892028808594, -53.477542877197266, -76.0671157836914, -309.77105712890625, -92.77632141113281, -250.01596069335938, -333.11029052734375, 42.17925262451172, -8.718254089355469, -137.56298828125, -190.11727905273438, -108.25469207763672, -78.81807708740234, -81.62205505371094, -10.541549682617188, -87.1081314086914, 77.39439392089844, 104.57418060302734, 71.57393646240234, -132.62069702148438, -123.30123138427734, 86.97282409667969, -39.40855407714844, -8.630341529846191, -498.3990478515625, -540.0208129882812, 67.42964172363281, 65.91172790527344, 278.6863708496094, 144.2326202392578, 44.34775924682617, 3.272491216659546, -285.34808349609375, 176.6351776123047, -120.71720123291016, 100.49688720703125, 40.00832748413086, 54.419403076171875, 178.53504943847656, -97.97322082519531, -77.44703674316406, -171.17230224609375, -75.43040466308594, 333.1881408691406, -72.66373443603516, 237.12326049804688, 165.20228576660156, 145.85635375976562, 160.60302734375, -202.85531616210938, 74.68953704833984, 317.68243408203125, -139.1591796875, -9.421417236328125, -137.8318328857422, 63.40230178833008, -215.70602416992188, 321.9359436035156, -75.36695861816406, -308.5323181152344, 358.1581726074219, 194.10458374023438, 103.7022933959961, 106.51750946044922, -71.55960083007812, -44.65911865234375, 144.80384826660156, 183.52499389648438, 186.52247619628906, 42.510196685791016, -93.27761840820312, -38.41746520996094, 221.91993713378906, 287.9836120605469, -68.26248168945312, -95.46080017089844, 249.5099334716797, 144.72979736328125, 139.45164489746094, -262.46673583984375, -74.7514877319336, -102.99910736083984, 41.5809326171875, -336.9884033203125, 254.35055541992188, 60.02635955810547, 86.9674072265625, -236.8280792236328, -5.284074306488037, 266.9300537109375, -313.9391174316406, 61.15361022949219, 139.760986328125, -138.38941955566406, 162.9482421875, -66.01380920410156, 245.95330810546875, 105.50235748291016, 44.87317657470703, 401.783203125, 166.37928771972656, 261.7271423339844, -237.9039306640625, -129.3214111328125, 167.3137969970703, 181.12237548828125, -11.148006439208984, 155.9453125, -211.67074584960938, 42.76704406738281, -229.45677185058594, 68.11410522460938, -97.23438262939453, -128.1995086669922, 11.259805679321289, 363.49200439453125, -202.3523406982422, -120.3501205444336, 213.53282165527344, -84.71333312988281, 353.85284423828125, 44.50593948364258, 18.12638282775879, 71.3768310546875, 348.2359313964844, 131.0723114013672, 103.3424301147461, 50.371551513671875, -312.4837646484375, -177.05441284179688, 296.6688232421875, 164.37567138671875, 101.21297454833984, 127.41254425048828, -108.71861267089844, 67.59284973144531, -180.15892028808594, 78.2724609375, -142.8270263671875, -42.76396560668945, 90.68296813964844, -65.3575439453125, 129.51638793945312, 91.20765686035156, 71.8916244506836, 264.88226318359375, -294.7769470214844, -319.2564697265625, 123.24835968017578, 188.1588134765625, 402.3280944824219, 151.25135803222656, 166.04299926757812, -176.6160125732422, 150.52435302734375, 134.76779174804688, -211.17303466796875], "dtype": "float32"}], "e": {"__nd__": [-1.0, 1.0, -1.0, -1.0, -1.0, 1.0, 1.0, -1.0, 1.0, 1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, 1.0, -1.0, -1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -0.9999999403953552, -1.0, 1.0, -1.0, -1.0, 1.0, 0.9999996423721313, 1.0, 1.0, 1.0, 1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, 1.0, -0.9999999403953552, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, 1.0, 1.0, 1.0, -1.0, -1.0, 1.0, -1.0, -0.9999999403953552, -1.0, -1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.997129499912262, -1.0, 1.0, -1.0, 1.0, 1.0, 1.0, 1.0, -1.0, -1.0, -1.0, -1.0, 1.0, -1.0, 1.0, 1.0, 1.0, 1.0, -1.0, 1.0, 1.0, -1.0, -1.0, -1.0, 1.0, -1.0, 1.0, -1.0, -1.0, 1.0, 1.0, 1.0, 1.0, -1.0, -1.0, 1.0, 1.0, 1.0, 1.0, -1.0, -1.0, 1.0, 1.0, -1.0, -1.0, 1.0, 1.0, 1.0, -1.0, -1.0, -1.0, 1.0, -1.0, 1.0, 1.0, 1.0, -1.0, -0.9999485611915588, 1.0, -1.0, 1.0, 1.0, -1.0, 1.0, -1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, -1.0, -1.0, 1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, -1.0, 1.0, 1.0, -1.0, -1.0, 1.0, -1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, -1.0, -1.0, 1.0, 1.0, 1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, -1.0, 1.0, -1.0, 1.0, 1.0, 1.0, 1.0, -1.0, -1.0, 1.0, 1.0, 1.0, 1.0, 1.0, -1.0, 1.0, 1.0, -1.0], "dtype": "float32"}}]'
import json, types
harness = types.ModuleType('harness'); exec(_HARNESS, harness.__dict__)
res = harness.grade(tanh, json.loads(_TESTS), 'cuda', *[1e-05, 1e-06])
for i, r in enumerate(res): print(f"test {i+1}: {'pass' if r['ok'] else 'FAIL  ' + r['why']}")
print(f"{sum(r['ok'] for r in res)}/{len(res)} passed")